# TreeHeight

[Codility task](https://app.codility.com/programmers/trainings/4/tree_height/) - Exercise 4, Algorithmic skills

The **height** of a binary tree is the number of pointers on its longest downward path: a single node has height `0` and, by convention, the empty tree has height `-1`.

`solution(T)` returns the height of a non-empty tree. Example:

```
(5, (3, (20, None, None), (21, None, None)), (10, (1, None, None), None))   ->  2
```

Assumptions: `N` in `[1..1,000]`, height in `[0..500]`.

In [ ]:
from dataclasses import dataclass


@dataclass
class Tree:
    x: int = 0
    l: "Tree" = None
    r: "Tree" = None


def from_tuple(t):
    """Build a Tree from the (X, L, R) notation of the statement (None = empty)."""
    if t is None:
        return None
    x, l, r = t
    return Tree(x, from_tuple(l), from_tuple(r))

## Idea

Recursive definition: `height(None) = -1`, `height(node) = 1 + max(height(l), height(r))`.

The height can be 500 and Python's recursion limit is 1000, so the recursive version barely fits. The **iterative** version walks the tree level by level (BFS) and counts levels: the height is the number of levels minus one.

In [ ]:
def solution(T):
    """Iterative level-order traversal."""
    height = -1
    level = [T]
    while level:
        height += 1
        next_level = []
        for node in level:
            if node.l is not None:
                next_level.append(node.l)
            if node.r is not None:
                next_level.append(node.r)
        level = next_level
    return height


def solution_recursive(T):
    """Direct translation of the definition (fine for height <= ~900)."""
    if T is None:
        return -1
    return 1 + max(solution_recursive(T.l), solution_recursive(T.r))

## Official example

In [ ]:
example = from_tuple((5, (3, (20, None, None), (21, None, None)), (10, (1, None, None), None)))
assert solution(example) == 2
assert solution_recursive(example) == 2
print("example OK")

## Edge cases

In [ ]:
assert solution(from_tuple((1, None, None))) == 0                          # single node
assert solution(from_tuple((1, (2, None, None), None))) == 1               # one edge, left
assert solution(from_tuple((1, None, (2, None, None)))) == 1               # one edge, right
assert solution(from_tuple((1, (2, None, None), (3, None, None)))) == 1    # balanced, 3 nodes
assert solution(from_tuple((1, (2, (3, (4, None, None), None), None), (5, None, None)))) == 3  # unbalanced
assert solution_recursive(from_tuple((1, (2, (3, (4, None, None), None), None), (5, None, None)))) == 3
print("edge cases OK")

## Brute-force check

The reference enumerates every root-to-node path with an explicit stack and keeps the longest one. Random trees are built by attaching new nodes to random free child slots.

In [ ]:
import random


def longest_path(T):
    best = 0
    stack = [(T, 0)]
    while stack:
        node, length = stack.pop()
        best = max(best, length)
        for child in (node.l, node.r):
            if child is not None:
                stack.append((child, length + 1))
    return best


def random_tree(n, rng):
    root = Tree(0)
    free = [(root, "l"), (root, "r")]
    for i in range(1, n):
        parent, side = free.pop(rng.randrange(len(free)))
        node = Tree(i)
        setattr(parent, side, node)
        free += [(node, "l"), (node, "r")]
    return root


rng = random.Random(3)
for _ in range(3000):
    tree = random_tree(rng.randint(1, 40), rng)
    expected = longest_path(tree)
    assert solution(tree) == expected
    assert solution_recursive(tree) == expected
print("both solutions match the reference on 3000 random trees")

## Extreme trees

- a chain of height 500 (N = 501) and a chain with N = 1000;
- a perfectly balanced tree with 1,023 nodes: height 9.

In [ ]:
def chain(height, side="l"):
    root = cur = Tree(0)
    for _ in range(height):
        child = Tree(0)
        setattr(cur, side, child)
        cur = child
    return root


def complete(depth):
    return None if depth == 0 else Tree(0, complete(depth - 1), complete(depth - 1))


assert solution(chain(500)) == 500
assert solution_recursive(chain(500)) == 500
assert solution(chain(999, "r")) == 999
assert solution(complete(10)) == 9            # 2**10 - 1 = 1023 nodes
print("extreme trees OK")

## Notes

- `-1` for the empty tree is what makes the recursion `1 + max(...)` correct for leaves.
- Python raises `RecursionError` near depth 1000; the iterative BFS has no such limit, which matters for degenerate (list-like) trees.
- Never compare or print large trees: the `@dataclass`-generated `__eq__` and `__repr__` are recursive.